In [ ]:
# imports 

import os
import numpy as np
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates

import tifffile
from scipy.optimize import curve_fit

from vti_magnetic_import import load_vti_sample, inspect_vti

In [ ]:
def plot_energy_maps_fancy(data_dict, key, axis='z', index=None, cmap='viridis', vmin=None, vmax=None, voxel_nm = 8.0, title = ''):
    """
    """
    nz, ny, nx = data_dict['shape']
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    print(index)
    if index is None:
        index = default_index[axis]

    data = data_dict[key]
    slice_x = data[:, :, default_index['x']]   # (nz, ny) -- y-z plane at fixed x
    slice_y = data[:, default_index['y'], :]   # (nz, nx) -- x-z plane at fixed y
    slice_z = data[index, :, :]   # (ny, nx) -- x-y plane at fixed z


    LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 16}
    TITLE_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}
    TICK_FONT = {'family': 'Times New Roman', 'size': 13}
    CBAR_LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}

    def style_axis(ax):
        ax.set_xlabel(ax.get_xlabel(), fontdict=LABEL_FONT)
        ax.set_ylabel(ax.get_ylabel(), fontdict=LABEL_FONT)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])

    def style_colorbar(cbar, label_text):
        cbar.set_label(label_text, fontdict=CBAR_LABEL_FONT)
        cbar.formatter = mticker.ScalarFormatter(useMathText=True)
        cbar.formatter.set_powerlimits((0, 0))
        cbar.update_ticks()
        for label in cbar.ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])
        cbar.ax.yaxis.get_offset_text().set_fontname('Times New Roman')
        cbar.ax.yaxis.get_offset_text().set_fontsize(TICK_FONT['size'])

    # Conversion factor: 1 pix = 8 nm
    PIX_TO_NM = voxel_nm #2 #8


    # Calculate extents in nm
    extent_x = [0, ny * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]  # y, z
    extent_y = [0, nx * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]  # x, z
    extent_z = [0, nx * PIX_TO_NM / 1000, 0, ny * PIX_TO_NM / 1000]  # x, y


    # Create figure with GridSpec for custom layout
    fig = plt.figure(figsize=(25,8)) ##################################################################################
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 2, 0.15], height_ratios=[1, 1], 
                        hspace=0.37, wspace=0.15)

    # Z slice (left, full square)
    ax_z = fig.add_subplot(gs[:, 0])

    vmax = np.max(slice_z)
    vmin = np.min(slice_z)

    im_z = ax_z.imshow(slice_z, origin='lower', aspect='equal', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_z)
    ax_z.set_title(f"Z slice", fontdict=TITLE_FONT, color='lime')
    ax_z.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_z.set_ylabel('y (μm)', fontdict=LABEL_FONT)
    ax_z.axhline(default_index['y'] * PIX_TO_NM/1000, color='magenta', lw=5, linestyle='--', zorder=3)
    ax_z.axvline(default_index['x'] * PIX_TO_NM/1000, color='darkorange', lw=5, linestyle='--', zorder=3)
    style_axis(ax_z)

    # X slice (top right, thin and long)
    ax_x = fig.add_subplot(gs[0, 1])
    im_x = ax_x.imshow(slice_x, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_x)
    ax_x.axhline(index* PIX_TO_NM + PIX_TO_NM/2, color='lime', lw=5, linestyle='--', zorder=3)
    ax_x.set_title(f"X slice", fontdict=TITLE_FONT, color='darkorange')
    ax_x.set_xlabel('y (μm)', fontdict=LABEL_FONT)
    ax_x.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_x)

    # Y slice (bottom right, thin and long)
    ax_y = fig.add_subplot(gs[1, 1])
    im_y = ax_y.imshow(slice_y, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_y)
    ax_y.axhline(index* PIX_TO_NM + PIX_TO_NM/2, color='lime', lw=5, linestyle='--', zorder=3)
    ax_y.set_title(f"Y slice", fontdict=TITLE_FONT, color='magenta')
    ax_y.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_y.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_y)


    # Colorbar for Z slice
    cbar_xy = fig.colorbar(im_x, ax=[ax_x, ax_y], orientation='vertical', 
                            pad=0.05, aspect=30)
    style_colorbar(cbar_xy, key)

    if index is None:
        fig.suptitle(f"{key} Contrast at Volume Center", 
                    fontfamily=LABEL_FONT['family'],
                    fontweight=LABEL_FONT['weight'],
                    fontsize=25)

    else:
        fig.suptitle(f"{key} Contrast at thickness = {index*voxel_nm} nm", 
                    fontfamily=LABEL_FONT['family'],
                    fontweight=LABEL_FONT['weight'],
                    fontsize=25)
    #plt.show()

    # fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    # for ax, key in zip(axes, keys):
    #     data = data_dict[key]
    #     img, xlabel, ylabel, extent = get_slice(data, axis, index, nz, ny, nx)

    #     im = ax.imshow(img, origin='lower', aspect='auto', cmap=panel_cmap[key], extent=extent)
    #     ax.set_title(key)
    #     ax.set_xlabel(xlabel)
    #     ax.set_ylabel(ylabel)
    #     fig.colorbar(im, ax=ax, label=key)

    # fig.suptitle(f"{title} for slice {axis} at index {index}")
    # plt.tight_layout()
    # plt.show()
    return #fig

# Make simulated bloch walls

In [ ]:
def make_stripe_volume(nx=100, ny=100, nz=7, period=25.0, dw_width=2.0, angle_deg=0.0):
    """
    PMA stripe domains with Bloch walls, identical in every z layer.
    Arrays are indexed (z, y, x).

    period    : full stripe period (up + down domain) in pixels
    dw_width  : wall width parameter delta in pixels, Mz = tanh(x/delta)
    angle_deg : in-plane direction of the wall normal (0 -> wall normal along x, stripes along y)
    """
    th = np.deg2rad(angle_deg)
    y, x = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')

    # coordinate perpendicular to the stripes
    u = x * np.cos(th) + y * np.sin(th)

    # nearest wall: walls sit at period/4 + n*period/2
    n = np.round((u - period / 4) / (period / 2))
    dist = u - (period / 4 + n * (period / 2))
    s = (-1.0) ** n   # alternates: up->down wall, then down->up wall

    Mz_2d = -s * np.tanh(dist / dw_width)
    #Mw_2d = s * (1.0 / np.cosh(dist / dw_width))   # in-wall component, along the stripe direction
    Mw_2d = 1.0 / np.cosh(dist / dw_width)

    # Bloch wall: in-plane component points along the stripe (perpendicular to the wall normal)
    Mx_2d = -np.sin(th) * Mw_2d
    My_2d = np.cos(th) * Mw_2d

    # same in every z layer
    Mx = np.repeat(Mx_2d[None], nz, axis=0)
    My = np.repeat(My_2d[None], nz, axis=0)
    Mz = np.repeat(Mz_2d[None], nz, axis=0)

    magnitude = np.sqrt(Mx**2 + My**2 + Mz**2)
    return {'Mx': Mx, 'My': My, 'Mz': Mz, 'magnitude': magnitude, 'shape': Mx.shape}


def save_sample_data(cropped, save_dir, sample_name, date, spacing=(1.0, 1.0, 1.0)):
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, f'{sample_name}_volume_{date}.npz')
    np.savez(path,
             Mx=cropped['Mx'], My=cropped['My'], Mz=cropped['Mz'],
             magnitude=cropped['magnitude'],
             shape=np.array(cropped['shape']),
             spacing=np.array(spacing))
    return path

#p = save_sample_data(cropped, save_dir, f'stripes_nz{nz}_ang{angle}', date)


def make_stripe_volume_psi(psi_deg, nx=100, ny=100, period=25, dw_width=2.0, angle_deg=0.0):
    """
    PMA stripe domains with a wall type set per z layer. Arrays are indexed (z, y, x).

    psi_deg   : array of length nz, wall angle for each z layer (z index 0 = bottom, nz-1 = top)
                0 = Bloch, +90 = Neel, -90 = Neel with opposite chirality, in between = hybrid
    period    : full stripe period in pixels
    dw_width  : wall width parameter delta in pixels
    angle_deg : in-plane direction of the wall normal (0 -> wall normal along x)
    """
    psi = np.deg2rad(np.asarray(psi_deg, dtype=float))
    nz = len(psi)
    th = np.deg2rad(angle_deg)

    y, x = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
    u = x * np.cos(th) + y * np.sin(th)   # coordinate along the wall normal

    # wall positions (same construction as before)
    n_hi = int(np.ceil((u.max() - period / 4) / (period / 2))) + 2
    n_lo = int(np.floor((u.min() - period / 4) / (period / 2))) - 2
    if n_lo % 2:
        n_lo -= 1
    wall_pos = period / 4 + np.arange(n_lo, n_hi + 1) * (period / 2)

    phi = (2 * np.arctan(np.exp((u[..., None] - wall_pos) / dw_width))).sum(axis=-1)
    Mz_2d = np.cos(phi)
    Mw_2d = np.sin(phi)   # in-plane magnitude, smooth through zero at domain centers

    # unit vectors in the plane: t along the stripe, n along the wall normal
    tx, ty = -np.sin(th), np.cos(th)
    nx_, ny_ = np.cos(th), np.sin(th)

    c = np.cos(psi)[:, None, None]
    s = np.sin(psi)[:, None, None]

    # in-plane component = Mw * (cos(psi) * t + sin(psi) * n)
    Mx = Mw_2d[None] * (c * tx + s * nx_)
    My = Mw_2d[None] * (c * ty + s * ny_)
    Mz = np.repeat(Mz_2d[None], nz, axis=0)

    magnitude = np.sqrt(Mx**2 + My**2 + Mz**2)
    return {'Mx': Mx, 'My': My, 'Mz': Mz, 'magnitude': magnitude, 'shape': Mx.shape}


def make_stripe_volume_neel(nz=5, **kw):
    """100% Neel walls in every layer, same chirality through the thickness."""
    return make_stripe_volume_psi(np.full(nz, 90.0), **kw)


def make_stripe_volume_twisted(nz=5, **kw):
    """Top = Neel (+90), middle = Bloch (0), bottom = opposite-chirality Neel (-90),
    with a linear twist of psi through the thickness."""
    return make_stripe_volume_psi(np.linspace(-90.0, 90.0, nz), **kw)

In [ ]:
def make_stripe_volume_new(nx=101, ny=101, nz=5, period=20, dw_width=2.0, angle_deg=0.0):
    """
    PMA stripe domains with Bloch walls, identical in every z layer.
    Arrays are indexed (z, y, x).
    All walls have the same chirality (defined for +Mz -> -Mz), and the
    in-wall component passes smoothly through zero at domain centers.

    period    : full stripe period (up + down domain) in pixels
    dw_width  : wall width parameter delta in pixels
    angle_deg : in-plane direction of the wall normal (0 -> wall normal along x, stripes along y)
    """
    th = np.deg2rad(angle_deg)
    y, x = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')

    # coordinate perpendicular to the stripes
    u = x * np.cos(th) + y * np.sin(th)

    # wall positions: period/4 + n*period/2; even n = up->down, odd n = down->up
    # n_lo is even so the first domain in the box is "up"
    n_hi = int(np.ceil((u.max() - period / 4) / (period / 2))) + 2
    n_lo = int(np.floor((u.min() - period / 4) / (period / 2))) - 2
    if n_lo % 2:
        n_lo -= 1
    wall_pos = period / 4 + np.arange(n_lo, n_hi + 1) * (period / 2)

    # rotation angle of M: each wall adds pi
    phi = (2 * np.arctan(np.exp((u[..., None] - wall_pos) / dw_width))).sum(axis=-1)

    Mz_2d = np.cos(phi)
    Mw_2d = np.sin(phi)   # in-wall component, along the stripe direction

    # Bloch wall: in-plane component points along the stripe
    Mx_2d = -np.sin(th) * Mw_2d
    My_2d = np.cos(th) * Mw_2d

    Mx = np.repeat(Mx_2d[None], nz, axis=0)
    My = np.repeat(My_2d[None], nz, axis=0)
    Mz = np.repeat(Mz_2d[None], nz, axis=0)

    magnitude = np.sqrt(Mx**2 + My**2 + Mz**2)
    return {'Mx': Mx, 'My': My, 'Mz': Mz, 'magnitude': magnitude, 'shape': Mx.shape}

In [ ]:
# set up simulation
date = '18NOV26'
nz = 5
wall_thickness_px = 2 #px

save_dir = f'stripe_dw_{wall_thickness_px}_{nz}z'

# volume_gamma0 =  make_stripe_volume_new(nz=nz, angle_deg=0)
# volume_gamma45 =  make_stripe_volume_new(nz=nz, angle_deg=45)
# volume_gamma90 =  make_stripe_volume_new(nz=nz, angle_deg=90)


In [ ]:
vol_neel = make_stripe_volume_neel(nz=nz, nx=99, ny=99, period=20, dw_width=2.0, angle_deg=0.0)
vol_twist = make_stripe_volume_twisted(nz=nz, nx=99, ny=99, period=20, dw_width=2.0, angle_deg=0.0)

print(np.abs(vol_neel['magnitude'] - 1).max(), np.abs(vol_twist['magnitude'] - 1).max())

# for angle_deg=0, Neel in-plane = Mx, Bloch in-plane = My
for z in range(nz):
    print(z, 'max|Mx| =', np.abs(vol_twist['Mx'][z]).max().round(3),
             'max|My| =', np.abs(vol_twist['My'][z]).max().round(3))

In [ ]:
plot_energy_maps_fancy(vol_neel, 'Mx', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(vol_twist, 'Mx', voxel_nm = 1)


In [ ]:
save_dir

In [ ]:
# Save tiff of each component for Dinesh's forward projection

def save_components_tiff(vol, save_dir, name):
    """Save Mx, My, Mz as separate float32 TIFF stacks, shape (nz, ny, nx)."""
    os.makedirs(save_dir, exist_ok=True)
    paths = []
    for comp in ('Mx', 'My', 'Mz'):
        path = os.path.join(save_dir, f'{name}_{comp}.tiff')
        tifffile.imwrite(path, vol[comp].astype(np.float32))
        paths.append(path)
    return paths


# paths = save_components_tiff(vol_neel, save_dir, f'neel_stripes_wall_{wall_thickness_px}px_t_{nz}z')
# paths = save_components_tiff(vol_twist, save_dir, f'twist_stripes_wall_{wall_thickness_px}px_t_{nz}z')
# print(paths)

In [ ]:
import os
import numpy as np
import pyvista as pv

def save_vol_vti(vol, save_dir, name, spacing=(1.0, 1.0, 1.0)):
    nz, ny, nx = vol['Mz'].shape

    # VTK wants x varying fastest, which matches C-order flattening of a (z, y, x) array
    vectors = np.stack([vol['Mx'], vol['My'], vol['Mz']], axis=-1).reshape(-1, 3)

    grid = pv.ImageData(dimensions=(nx, ny, nz), spacing=spacing, origin=(0, 0, 0))
    grid.point_data['M'] = vectors.astype(np.float32)
    grid.point_data['Mx'] = vol['Mx'].ravel().astype(np.float32)
    grid.point_data['My'] = vol['My'].ravel().astype(np.float32)
    grid.point_data['Mz'] = vol['Mz'].ravel().astype(np.float32)

    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, f'{name}.vti')
    grid.save(path)
    return path

print(save_vol_vti(vol_neel, save_dir, 'neel_stripes_nz5_ang0_99'))
print(save_vol_vti(vol_twist, save_dir, 'twist_stripes_nz5_ang0_99'))

# load reconstructed volume and compare 

In [ ]:
def load_sample_data(save_dir, sample_name, date):
    """
    Load back the cropped volume for one sample
    """
    volume_path = os.path.join(save_dir, f'{sample_name}_volume_{date}.npz')

    npz = np.load(volume_path)
    cropped = {
        'Mx': npz['Mx'], 'My': npz['My'], 'Mz': npz['Mz'],
        'magnitude': npz['magnitude'],
        'shape': tuple(npz['shape']),
    }
    spacing = tuple(npz['spacing'])
    z_offset = 0

    return cropped, spacing, z_offset

def get_slice(data, axis, index, nz, ny, nx):
    if axis == 'x':
        return data[:, :, index], 'y (voxel)', 'z (voxel)', [0, ny, 0, nz]
    elif axis == 'y':
        return data[:, index, :], 'x (voxel)', 'z (voxel)', [0, nx, 0, nz]
    elif axis == 'z':
        return data[index, :, :], 'x (voxel)', 'y (voxel)', [0, nx, 0, ny]
    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")

In [ ]:
sample_names = ['452'] 
dates = ['sim_stripe_TV_2px_5z_NOV26_v2'] 
sample_colors = ['limegreen']


sample = 0
sample_name = sample_names[sample]
date = dates[sample]
sample_color = sample_colors[sample]


data_dir = f'C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/projections/simulation/'

recon_vol, spacing, ___ = load_sample_data(data_dir, sample_name, date)
print(spacing)
voxel_nm = spacing[0]

In [ ]:
plot_energy_maps_fancy(recon_vol, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(recon_vol, 'My', voxel_nm = 1)
plot_energy_maps_fancy(recon_vol, 'Mx', voxel_nm = 1)

In [ ]:
stop

In [ ]:
plot_energy_maps_fancy(volume_gamma0, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma0, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma0, 'Mx', voxel_nm = 1)



In [ ]:
plot_energy_maps_fancy(volume_gamma45, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma45, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma45, 'Mx', voxel_nm = 1)

In [ ]:
plot_energy_maps_fancy(volume_gamma90, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma90, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma90, 'Mx', voxel_nm = 1)

# project

In [ ]:
# upload experimental angles: 

def project_xmcd(My, Mz, alpha_deg):
    """
    XMCD projection at one tilt angle about the x axis.
    Beam direction k = (0, sin a, cos a) in (x, y, z).
    Signal = line integral of M.k along the beam (1 px steps).
    Returns a (ny, nx) image. My does not contribute for a y tilt axis.
    """
    nz, ny, nx = My.shape
    # print(nz, ny, nx)
    a = np.deg2rad(alpha_deg)
    s, c = np.sin(a), np.cos(a)

    vol = My * s + Mz * c   # scalar M.k at every voxel

    # detector coordinates (centered) and ray parameter t along the beam
    cx, cy, cz = (nx - 1) / 2, (ny - 1) / 2, (nz - 1) / 2
    half = int(np.ceil(0.5 * np.sqrt(ny**2 + nz**2)))
    t = np.arange(-half, half + 1)
    YP, X, T = np.meshgrid(np.arange(ny) - cy, np.arange(nx) - cx, t, indexing='ij')

    # point on the ray: x' * e1 + y * e2 + t * k, with e1 = (cos a, 0, -sin a)
    y = YP * c + T * s
    z = -YP * s + T * c

    coords = [z + cz, y + cy, X + cx]
    samples = map_coordinates(vol, coords, order=1, mode='constant', cval=0.0)
    return samples.sum(axis=-1) / nz


def make_tilt_series(My, Mz, angles_deg):
    return np.stack([project_xmcd(My, Mz, a) for a in angles_deg])   # (n_angles, ny, nx)


def show_tilt_series(tilt_series, angles, n_show=8):
    idx = np.linspace(0, len(angles) - 1, n_show).astype(int)
    vmax = np.abs(tilt_series).max()

    fig, axs = plt.subplots(2, n_show // 2, figsize=(3 * n_show // 2, 6))
    for ax, i in zip(axs.ravel(), idx):
        im = ax.imshow(tilt_series[i], cmap='viridis', vmin=-vmax, vmax=vmax, origin='lower')
        ax.set_title(f'{angles[i]:.1f}°')
        ax.axis('off')
    fig.colorbar(im, ax=axs, shrink=0.8, label='projected M·k (sum over beam path)')
    plt.show()

def show_line_cuts(tilt_series, angles, row=None):
    row = tilt_series.shape[1] // 2 if row is None else row
    for img, a in zip(tilt_series, angles):
        plt.plot(img[row], label=f'{a:.0f}°')
    plt.xlabel('x (px)')
    plt.ylabel('projected signal')
    plt.legend(ncol=2, fontsize=7)
    plt.show()



# 452

In [ ]:
sample_name = '452'
nz = 5


volume_gamma1 =  make_stripe_volume_new(nz=nz, angle_deg=-0.07)
volume_gamma51 =  make_stripe_volume_new(nz=nz, angle_deg=49.89)
volume_gamma79 =  make_stripe_volume_new(nz=nz, angle_deg=77.8)

plot_energy_maps_fancy(volume_gamma1, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma1, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma1, 'Mx', voxel_nm = 1)

plot_energy_maps_fancy(volume_gamma51, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma51, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma51, 'Mx', voxel_nm = 1)

plot_energy_maps_fancy(volume_gamma79, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma79, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma79, 'Mx', voxel_nm = 1)

In [ ]:

# angles from alignment files
angle_1 = np.array([ 29.35698122, 33.79176380, 38.22376783, 39.55530396, 40.88432756, 43.54069002, 44.87387692, 46.20211539, 47.53174149, 48.85983102, 52.84657552, 54.17436719, 55.50305913, 56.83216237, 59.48868576, 60.81707956, 64.80181176, 66.13039406, 67.45841515, 68.78538755])
angle_51 = np.array([-68.09173441, -68.08879370, -66.75513429, -64.52865439, -62.74638394, -61.85359544, -60.96574872, -59.18430054, -58.29265975, -57.40350920, -55.62239363, -54.73042890, -53.84144440, -52.06115087, -48.94371157, -47.61041046, -46.27475452, -44.94017159, -43.60519299, -42.26883859, -40.93502540, -39.60081658, -38.26376513, -36.92925352, -28.03245799, -23.58733170, -23.58635341, -19.14275621, -14.69970872, -10.25721227, -5.81575540, 3.06648141, 7.50530683, 11.94162927, 16.37837544, 20.81408279, 25.24874935, 29.68237711, 34.11350160, 38.54602417, 42.97701956, 44.30582343, 48.28972353, 49.62058807, 50.94843738, 52.27716558, 53.60336588, 54.93336883, 56.26084403, 57.58919884, 58.91599848, 60.24465360, 65.11444861, 66.43731204, 67.76835638, 69.09444010, 70.42140283])
angle_79 = np.array([-68.93715437, -67.60760054, -66.27514515, -64.94223259, -63.61130596, -62.27943290, -60.94807906, -59.61675621, -58.28595354, -56.95371535, -55.62150921, -52.96109709, -51.62458694, -50.29396931, -48.95849871, -47.62305932, -46.28765113, -44.95666968, -43.62327699, -42.29186956, -37.85048751, -28.97071082, -24.53524620, -15.66046931, -11.22310888, -6.78707086, -2.34649668, 2.09129168, 6.52629577, 10.96193024, 15.39819585, 19.83460389, 24.27017929, 28.70297045, 33.13834325, 38.90434815, 40.23383232, 41.56377180, 44.22404691, 45.55340595, 46.88419680, 48.21300650, 49.54422247, 50.87297011, 52.20266222, 54.86146410, 56.19155028, 57.52014284, 58.85016692, 60.17967277, 61.50817129, 62.83712793, 64.16800207, 65.49738374, 66.82722147, 68.15556487])



proj_1 = make_tilt_series(volume_gamma1['My'], volume_gamma1['Mz'], angle_1)

show_tilt_series(proj_1, angle_1)
# show_line_cuts(proj_1, angle_1)

p0 = project_xmcd(volume_gamma1['My'], volume_gamma1['Mz'], 0.0)
print(np.abs(p0 - volume_gamma1['Mz'].sum(axis=0)).max())   # should be ~1e-15

In [ ]:
print(proj_1.shape)

In [ ]:
proj_51 = make_tilt_series(volume_gamma51['My'], volume_gamma51['Mz'], angle_51)

show_tilt_series(proj_51, angle_51)
# show_line_cuts(proj_51, angle_51)

# p0 = project_xmcd(volume_gamma51['Mx'], volume_gamma51['Mz'], 0.0)
# print(np.abs(p0 - volume_gamma51['Mz'].sum(axis=0)).max())   # should be ~1e-15



proj_79 = make_tilt_series(volume_gamma79['My'], volume_gamma79['Mz'], angle_79)

show_tilt_series(proj_79, angle_79)
# show_line_cuts(proj_79, angle_79)

# p0 = project_xmcd(volume_gamma79['Mx'], volume_gamma79['Mz'], 0.0)
# print(np.abs(p0 - volume_gamma79['Mz'].sum(axis=0)).max())   # should be ~1e-15

In [ ]:

save_dir = r'C:\Users\emmab\Projects\Vector_Tomography\data_dir\452\projections\simulation'

tiff_path = os.path.join(save_dir, f'{sample_name}_tilt1_NOV26.tiff')
tifffile.imwrite(tiff_path, proj_1.astype(np.float32))

angle1_path = os.path.join(save_dir, 'angle_1.txt')
np.savetxt(angle1_path, angle_1)



tiff_path = os.path.join(save_dir, f'{sample_name}_tilt51_NOV26.tiff')
tifffile.imwrite(tiff_path, proj_51.astype(np.float32))

angle1_path = os.path.join(save_dir, 'angle_51.txt')
np.savetxt(angle1_path, angle_51)



tiff_path = os.path.join(save_dir, f'{sample_name}_tilt79_NOV26.tiff')
tifffile.imwrite(tiff_path, proj_79.astype(np.float32))

angle1_path = os.path.join(save_dir, 'angle_79.txt')
np.savetxt(angle1_path, angle_79)


# compare recon to sim

In [ ]:

def wall_profile(Mz_slice, angle_deg, margin=8, bin_width=0.5):
    """Average Mz along the stripes -> 1D profile across the walls (binned in u)."""
    ny, nx = Mz_slice.shape
    th = np.deg2rad(angle_deg)
    y, x = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
    u = x * np.cos(th) + y * np.sin(th)

    # ignore the box edges, where the reconstruction has boundary artifacts
    keep = (x >= margin) & (x < nx - margin) & (y >= margin) & (y < ny - margin)
    u, v = u[keep], Mz_slice[keep]

    bins = np.floor((u - u.min()) / bin_width).astype(int)
    count = np.bincount(bins)
    ok = count > 0
    prof = np.bincount(bins, weights=v)[ok] / count[ok]
    centers = np.bincount(bins, weights=u)[ok] / count[ok]
    return centers, prof


def fit_wall_widths(u, prof, half_window=4.0):
    """Fit A*tanh((u-u0)/delta) around each zero crossing. Returns array of deltas."""
    def model(u, A, u0, d):
        return A * np.tanh((u - u0) / d)

    widths = []
    for i in np.where(prof[:-1] * prof[1:] < 0)[0]:
        u0 = u[i] - prof[i] * (u[i + 1] - u[i]) / (prof[i + 1] - prof[i])
        if u0 - half_window < u.min() or u0 + half_window > u.max():
            continue   # skip walls cut off by the edge margin
        sel = np.abs(u - u0) <= half_window
        A0 = np.sign(prof[i + 1] - prof[i])
        p, _ = curve_fit(model, u[sel], prof[sel], p0=[A0, u0, 2.0])
        widths.append(abs(p[2]))
    return np.array(widths)


def measure_width(Mz, angle_deg):
    z_mid = Mz.shape[0] // 2
    u, prof = wall_profile(Mz[z_mid], angle_deg)
    return fit_wall_widths(u, prof)


def load_recon_Mz(angle_deg):
    # replace with however you load your reconstruction for this orientation
    # must return Mz with shape (z, y, x), same as the ground truth
    raise NotImplementedError




In [ ]:
data_dir = f"C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/projections/simulation/"

test_filename = f"452_sim_flippedyz.vtk"


test_path = os.path.join(data_dir, test_filename)
print(test_path)
inspect_vti(test_path)

In [ ]:
# --- Cell: load the sample and summarize ---
samples = {}
samples[sample_name] = load_vti_sample(test_path, sample_name=sample_name)

s = samples[sample_name]
print("\nLoaded sample summary:")
for k, v in s.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {v}")

print("\nmagnitude min/max:", s["magnitude"].min(), s["magnitude"].max())

In [ ]:
plot_energy_maps_fancy(samples[sample_name], 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(samples[sample_name], 'My', voxel_nm = 1)
plot_energy_maps_fancy(samples[sample_name], 'Mx', voxel_nm = 1)

In [ ]:
plot_energy_maps_fancy(volume_gamma0, 'Mz', cmap = 'viridis', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma0, 'My', voxel_nm = 1)
plot_energy_maps_fancy(volume_gamma0, 'Mx', voxel_nm = 1)